In [3]:
!pip -q install groq gradio

In [11]:
import json, html, re
import gradio as gr
from groq import Groq
from google.colab import userdata

client = Groq(api_key="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")
MODEL = "openai/gpt-oss-120b"  # faster/cheaper: "openai/gpt-oss-20b"

SYSTEM = """You are a nutrition expert agent. Given a food, quantity and optional health note,
reply ONLY with JSON in this schema:
{"food": str, "edible": "yes"|"no"|"caution", "serving": str, "calories": number,
 "protein_g": number, "carbs_g": number, "fat_g": number, "fiber_g": number, "sugar_g": number,
 "vitamins_minerals": [str], "verdict": str, "reason": str, "tips": [str]}
Rules: if the item is not food or is unsafe/poisonous, set edible "no", numbers 0, and explain.
Use "caution" for foods that are risky in some cases (raw, allergens, health note conflict).
Account for the health note if given. Values are approximate for the stated quantity."""

COLORS = {"yes": ("#16a34a", "✅ Safe to eat"),
          "caution": ("#d97706", "⚠️ Eat with caution"),
          "no": ("#dc2626", "❌ Do not eat")}

def ask_model(user):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    try:
        r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0.2,
                                           response_format={"type": "json_object"})
    except Exception:  # fallback if JSON mode unsupported
        r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0.2)
    txt = r.choices[0].message.content
    m = re.search(r"\{.*\}", txt, re.S)
    return json.loads(m.group(0))

def analyze(food, qty, note):
    if not food.strip():
        return "<p>Enter a food name.</p>"
    user = f"Food: {food}\nQuantity: {qty or '1 standard serving'}\nHealth note: {note or 'none'}"
    try:
        d = ask_model(user)
    except Exception as ex:
        return f"<p style='color:red'>Error: {html.escape(str(ex))}</p>"

    e = lambda x: html.escape(str(x))
    color, label = COLORS.get(d.get("edible", "caution"), COLORS["caution"])
    chip = lambda k, v, u="g": f"<div class='chip'><b>{e(v)}{u}</b><br><small>{k}</small></div>"
    macros = "".join([chip("Protein", d.get("protein_g", 0)), chip("Carbs", d.get("carbs_g", 0)),
                      chip("Fat", d.get("fat_g", 0)), chip("Fiber", d.get("fiber_g", 0)),
                      chip("Sugar", d.get("sugar_g", 0))])
    vit = ", ".join(map(e, d.get("vitamins_minerals", []))) or "-"
    tips = "".join(f"<li>{e(t)}</li>" for t in d.get("tips", []))
    return f"""
<style>
  .nc, .nc * {{ color:#111 !important; }}
  .nc {{ background:#fff !important; border:2px solid {color}; border-radius:14px; padding:18px; font-family:sans-serif; }}
  .nc small {{ color:#666 !important; }}
  .nc .badge {{ background:{color} !important; color:#fff !important; display:inline-block; padding:4px 12px; border-radius:20px; margin:8px 0; }}
  .nc .badge * {{ color:#fff !important; }}
  .nc .row {{ display:flex; gap:8px; flex-wrap:wrap; }}
  .nc .chip {{ flex:1; min-width:90px; background:#f3f4f6 !important; border-radius:10px; padding:10px; text-align:center; }}
  .nc .chip b {{ font-size:18px; }}
  .nc h1, .nc h2 {{ margin:6px 0; }}
</style>
<div class='nc'>
  <h2>{e(d.get('food'))} <small>({e(d.get('serving'))})</small></h2>
  <div class='badge'>{label}</div>
  <h1>🔥 {e(d.get('calories'))} kcal</h1>
  <div class='row'>{macros}</div>
  <p><b>Vitamins & minerals:</b> {vit}</p>
  <p><b>Verdict:</b> {e(d.get('verdict'))}</p>
  <p><b>Why:</b> {e(d.get('reason'))}</p>
  <b>Tips:</b><ul>{tips}</ul>
  <small>AI estimates only, not medical advice.</small>
</div>"""

with gr.Blocks(theme=gr.themes.Soft(), title="Nutrition AI Agent") as demo:
    gr.Markdown("# 🥗 Nutrition AI Agent\nCalories, nutrients, and whether it's safe to eat. Powered by Groq.")
    with gr.Row():
        with gr.Column(scale=1):
            food = gr.Textbox(label="Food", placeholder="e.g. banana, pani puri, raw egg")
            qty = gr.Textbox(label="Quantity (optional)", placeholder="e.g. 2 pieces, 200 g")
            note = gr.Textbox(label="Health note (optional)", placeholder="e.g. diabetic, nut allergy")
            btn = gr.Button("Analyze", variant="primary")
            gr.Examples([["Banana", "1 medium", ""], ["White rice", "1 cup", "diabetic"],
                         ["Raw chicken", "100 g", ""], ["Peanut butter", "2 tbsp", "nut allergy"],
                         ["Bleach", "", ""]], [food, qty, note])
        with gr.Column(scale=2):
            out = gr.HTML()
    btn.click(analyze, [food, qty, note], out)
    food.submit(analyze, [food, qty, note], out)

demo.launch(share=True, debug=True)

/tmp/ipykernel_4932/1550153236.py:74: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Nutrition AI Agent") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ccdfe08e011baf8c02.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ccdfe08e011baf8c02.gradio.live
